# Lab 1.2. Object and field: from points to zonal statistics

**Module 2. Session 1: ACS-UPM Diploma in Engineering, Data Science and Artificial Intelligence**

In this lab, you will learn how to:

1. Read vector and raster data and recognise the data model used in each case.
2. Understand the role of the files that make up a shapefile.
3. Diagnose a missing or incorrect coordinate reference system (CRS).
4. Reproject geographic data before calculating areas or distances.
5. Convert a table of coordinates into a geographic table.
6. Sample a raster at specific points and summarise raster values within polygons.

### How to work

As in Lab 1.1, each exercise combines a short explanation, a task and a check.

Exercises marked **Extension** are optional.

Before finishing, restart the kernel and run the notebook from top to bottom to check that all cells execute correctly.


## 0. Environment

Run this cell first to check the versions of the main libraries used in the lab.


In [ ]:
import sys
import pandas as pd, numpy as np, geopandas as gpd, rasterio, matplotlib
print("Python    ", sys.version.split()[0])
for m in (pd, np, gpd, rasterio, matplotlib):
    print(f"{m.__name__:<10}", m.__version__)

## 0. Setup

Before starting, we need to locate the course folder and make sure the notebook can access the data files.

This setup cell is designed to work:

- on your own computer
- in Google Colab using the course repository
- in Google Colab using the shared Google Drive folder

Run the cell and check that the course folder, working folder and file list appear correctly.


In [ ]:
from pathlib import Path

REPO = "https://github.com/antiafer/acs-upm-mod2-s01.git"   # course repository
DRIVE = "ACS-UPM/Mod2-S01"                             # folder inside My Drive

def course_folder():
    """Return the folder that contains data/, wherever we are running."""
    here = Path.cwd()
    for base in (here, here.parent):                   # local clone
        if (base / "data").is_dir():
            return base
    try:
        import google.colab                            # noqa: F401
    except ImportError:
        raise FileNotFoundError("No data/ folder next to the notebook or one level up.")

    root = Path("/content/acs-mod2")                   # 1. try the repository
    if not (root / "data").is_dir():
        import subprocess
        subprocess.run(["git", "clone", "-q", REPO, str(root)], check=False)
    if (root / "data").is_dir():
        return root

    from google.colab import drive                     # 2. fall back to Drive
    if not Path("/content/drive").exists():
        drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive") / DRIVE
    if (root / "data").is_dir():
        return root
    raise FileNotFoundError(
        "Could not find the course folder. Either set REPO to the course repository, "
        f"or add a shortcut to the shared folder in My Drive as {DRIVE}.")

BASE = course_folder()
DATA = BASE / "data"
WORK = Path("/content") if Path("/content").exists() else Path.cwd()
print("Course folder:", BASE)
print("Working folder:", WORK)
sorted(p.name for p in DATA.iterdir())

`top10.parquet` is the output produced in Exercise 6 of Lab 1.1.

If the file is available in the current working folder, the next cell will reuse it. If not, it will rebuild the same table from the original source files.

This allows Lab 1.2 to run independently, even if you start a new Colab session.


In [ ]:
import matplotlib.pyplot as plt

def load_top10():
    """Read the output of Lab 1.1, or rebuild it from the source CSVs."""
    path = WORK / "top10.parquet"
    if path.exists():
        print("Reusing", path)
        return pd.read_parquet(path)
    print("Not found; rebuilding it from the source files.")
    kw = dict(sep=";", decimal=",", thousands=".", encoding="latin-1")
    counts = pd.read_csv(DATA / "aforos_202509.csv", parse_dates=["fecha"],
                         dayfirst=True, **kw).drop_duplicates()
    locations = pd.read_csv(DATA / "pm_ubicaciones.csv", **kw)
    means = (counts.groupby("id", as_index=False)["intensidad"].mean()
             .rename(columns={"intensidad": "mean_intensity"}))
    out = (means.sort_values("mean_intensity", ascending=False).head(10)
           .merge(locations, on="id", how="left").reset_index(drop=True))
    out.to_parquet(path, index=False)
    return out

top10 = load_top10()
print(len(top10), "measurement points carried over from Lab 1.1")

## 1. A shapefile is several files

A shapefile is not a single file. It is a set of related files that together store the geometry, attributes and supporting information of a vector layer.

The most common components are:

- `.shp` → geometry
- `.shx` → geometry index
- `.dbf` → attribute table
- `.prj` → coordinate reference system
- `.cpg` → text encoding

Some components, including `.prj` and `.cpg`, are optional. If they are missing, the geometry may still be readable while important information about how to interpret it is lost.

Inspect the files that make up the municipalities shapefile.


In [ ]:
shp_dir = DATA / "municipios_shp"
for p in sorted(shp_dir.iterdir()):
    print(f"{p.name:<20} {p.stat().st_size:>8} bytes")

In [ ]:
municipalities = gpd.read_file(shp_dir / "municipios.shp")
print("CRS:", municipalities.crs)
print("Geometry types:", municipalities.geometry.geom_type.unique())
print("Columns:", list(municipalities.columns))
municipalities.head(3)

Look at the name of the last attribute column.

In the original data it was `superficie_km2`, but in the shapefile it has been shortened. The dBase table used by the shapefile format traditionally limits field names to **10 characters**.

This is one common reason why attribute names in shapefiles may appear abbreviated or unclear.

Later, compare this with the same data stored as GeoJSON, where the full field name is preserved.


### Exercise 1. What happens when the `.prj` file is missing?

Create a copy of the shapefile in a `broken/` folder, but leave out the `.prj` file.

Then:

1. read the copied shapefile
2. inspect its CRS
3. restore the correct CRS, **EPSG:25830**

You will need to distinguish between two operations:

- `set_crs()` **declares** the CRS of coordinates that are already expressed in that system
- `to_crs()` **transforms** coordinates from one CRS to another

In this exercise, the coordinates have not changed; only the CRS information has been lost.


In [ ]:
import shutil
target = WORK / "broken"; target.mkdir(exist_ok=True)
for p in shp_dir.iterdir():
    if p.suffix != ".prj":
        shutil.copy(p, target / p.name)

# YOUR CODE HERE
raise NotImplementedError
print("CRS after losing the .prj:", lost_crs)
print("CRS after repairing:       ", repaired.crs)

In [ ]:
assert lost_crs is None, "Without a .prj the CRS should be None"
assert repaired.crs.to_epsg() == 25830
assert repaired.geometry.equals(municipalities.geometry), (
    "set_crs must not move any coordinate, only declare the system")
print("Checks passed.")

### Think about it

What would happen if you tried to use `to_crs(25830)` on the layer **before** declaring its original CRS?

Why is `set_crs(25830)` the appropriate operation in this case?

_Write your answer here:_


### Exercise 2. Areas should be calculated in an appropriate projected CRS

The municipalities are also provided as GeoJSON. Their coordinates are expressed in WGS 84 longitude and latitude, so the coordinate values are in degrees.

Read `municipios.geojson` and compare:

1. the result of calculating `.area` directly in the geographic CRS
2. the area calculated after reprojecting the layer to **EPSG:25830**, where coordinates are expressed in metres
3. the reference area stored in `superficie_km2`

The GeoJSON preserves the full field name `superficie_km2`, unlike the shapefile version.


In [ ]:
geo = gpd.read_file(DATA / "municipios.geojson")
print("CRS of the GeoJSON:", geo.crs)

# YOUR CODE HERE
raise NotImplementedError

comparison = pd.DataFrame({
    "municipality": geo["nombre"],
    "area_in_deg2": area_deg.round(6),
    "area_km2_computed": area_km2.round(3),
    "area_km2_declared": geo["superficie_km2"].round(3),
})
comparison

In [ ]:
rel_error = ((comparison["area_km2_computed"] - comparison["area_km2_declared"]).abs()
             / comparison["area_km2_declared"]).max()
assert geo.crs.to_epsg() == 4326, "The GeoJSON must be in WGS 84"
assert geo_utm.crs.to_epsg() == 25830
assert rel_error < 0.01, f"Relative area error too large: {rel_error:.3%}"
print(f"Checks passed. Maximum relative error: {rel_error:.4%}")

> **Working rule for this lab**
>
> Before calculating an area or distance:
>
> 1. check the CRS with `gdf.crs`
> 2. reproject to an appropriate projected CRS if necessary
> 3. calculate `area` or `length`
>
> We will study coordinate reference systems in more detail in Session 11.


### Exercise 3. From a table to a geographic table

`top10` is an ordinary pandas table. The columns `x_utm` and `y_utm` contain coordinates, but pandas treats them simply as numbers.

To use them as spatial data, we need to:

1. create point geometries from the coordinate columns
2. declare the CRS in which those coordinates are expressed
3. transform them when we need to combine them with data in another CRS

#### Step 1

The code below creates the points **without declaring their CRS** and plots them together with the municipalities, which are stored in geographic coordinates.

Before running it, make a prediction:

> **Do you expect the points and municipalities to overlap correctly? Why or why not?**


In [ ]:
points_nocrs = gpd.GeoDataFrame(
    top10, geometry=gpd.points_from_xy(top10["x_utm"], top10["y_utm"]))  # no crs

fig, ax = plt.subplots(figsize=(7, 5))
geo.plot(ax=ax, facecolor="#CFD8DC", edgecolor="white")
points_nocrs.plot(ax=ax, color="#E8590C", markersize=25)
ax.set_title("CRS mismatch: municipalities in degrees, points in metres")
plt.tight_layout()
print("CRS of the municipalities:", geo.crs)
print("CRS of the points        :", points_nocrs.crs)

The result shows the problem clearly.

The municipalities use longitude and latitude values around `-3.7` and `40.4`, while the traffic points use UTM coordinates around `440 000` and `4 480 000`.

Both layers can be plotted without an error, but their coordinates are expressed in different systems and units.

#### Step 2

Repair the points in two stages:

1. use `set_crs(25830)` to declare the CRS that the existing coordinates already use
2. use `to_crs(...)` to transform the points to the CRS of the municipalities

Store the declared layer in `points` and the transformed layer in `points_4326`.


In [ ]:
# YOUR CODE HERE
raise NotImplementedError

fig, ax = plt.subplots(figsize=(7, 5))
geo.plot(ax=ax, facecolor="#CFD8DC", edgecolor="white")
points_4326.plot(ax=ax, color="#E8590C", markersize=25)
for _, r in points_4326.iterrows():
    ax.annotate(r["nombre"], (r.geometry.x, r.geometry.y), fontsize=7, xytext=(4, 4),
                textcoords="offset points")
ax.set_title("After CRS declaration and transformation")
plt.tight_layout()

In [ ]:
assert points_nocrs.crs is None, "Step 1 should have left the points without a CRS"
assert points.crs.to_epsg() == 25830, "set_crs should have declared UTM 30N"
assert points_4326.crs == geo.crs, "to_crs should have moved the points to the municipalities' system"
assert points.geometry.equals(points_nocrs.geometry), "set_crs must not move any coordinate"
minx, miny, maxx, maxy = geo.total_bounds
inside = points_4326.geometry.x.between(minx, maxx) & points_4326.geometry.y.between(miny, maxy)
assert inside.all(), "After transforming, the points should fall inside the municipalities' bounding box"
print("Checks passed.")

> **Key point**
>
> When two spatial layers do not overlap as expected, check the CRS of both layers before changing the data.
>
> `set_crs()` and `to_crs()` solve different problems:
>
> - `set_crs()` assigns CRS information without changing coordinate values
> - `to_crs()` transforms coordinate values from one CRS to another
>
> Assigning the wrong CRS may not raise an immediate error, but subsequent spatial operations can produce incorrect results.


## 2. The field model

So far, we have worked mainly with **objects**: measurement points and municipalities with identifiable geometries and attributes.

Terrain elevation is different. It can be represented as a **field**, where a value is associated with every location across a continuous surface.

In GIS, these two models are commonly represented as:

- **vector data** for discrete objects
- **raster data** for fields or surfaces

### Exercise 4. Inspect the raster *(guided)*

This exercise is already solved.

Run the next cells and identify the main properties of the raster:

- CRS
- spatial resolution
- extent
- number of bands
- no-data value
- raster dimensions

You will use this information in the next exercise.


In [ ]:
src = rasterio.open(DATA / "mdt25_madrid.tif")
print("CRS       :", src.crs)
print("Resolution:", src.res, "m")
print("Extent    :", [round(v) for v in src.bounds])
print("Bands     :", src.count)
print("No-data   :", src.nodata)
print("Shape     :", src.shape, "->", src.shape[0] * src.shape[1], "cells")

In [ ]:
assert src.crs.to_epsg() == 25830
assert src.count == 1, "This terrain model is expected to have a single band"
assert src.nodata == -32768.0
assert src.res == (25.0, 25.0)
print("Checks passed.")

### No-data values in rasters

The raster uses `-32768` to represent cells where no elevation value is available.

This is another example of a **sentinel value**, similar to the `-1` used for missing traffic measurements in Lab 1.1.

If we plot the raster without handling this value, `-32768` is treated as a real elevation and distorts the colour scale.

The next cell compares the raster before and after replacing the no-data value with `NaN` for visualisation.


In [ ]:
elev = src.read(1)
masked = np.where(elev == src.nodata, np.nan, elev)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
im0 = axes[0].imshow(elev, cmap="terrain"); axes[0].set_title("unmasked")
plt.colorbar(im0, ax=axes[0], shrink=.8)
im1 = axes[1].imshow(masked, cmap="terrain"); axes[1].set_title("no-data set to NaN")
plt.colorbar(im1, ax=axes[1], shrink=.8)
for a in axes: a.set_axis_off()
plt.tight_layout()
print("Range unmasked:", np.nanmin(elev).round(1), "to", np.nanmax(elev).round(1))
print("Range masked  :", np.nanmin(masked).round(1), "to", np.nanmax(masked).round(1))

### Exercise 5. The question of the session

In Lab 1.1, we identified the ten traffic measurement points with the highest mean hourly intensity.

Now we can complete the spatial part of the question:

> **At what elevation are these ten measurement points?**

Sample the terrain raster at each point and add the result as a new column called `elev_m`.

`src.sample()` expects a sequence of `(x, y)` coordinates expressed in the **same CRS as the raster**.

Before sampling, check that the CRS of `points` matches the raster CRS.

> Spatial operations do not always detect a CRS mistake for you. The code may run while using coordinates in the wrong reference system.


In [ ]:
# YOUR CODE HERE
raise NotImplementedError

answer = (points[["nombre", "distrito", "mean_intensity", "elev_m"]]
          .round({"mean_intensity": 1, "elev_m": 1}))
answer

In [ ]:
assert "elev_m" in points.columns
assert points["elev_m"].notna().all(), "Some point fell in a no-data area"
assert points["elev_m"].between(400, 1200).all(), (
    f"Elevations outside a plausible range: {points['elev_m'].min():.0f} to {points['elev_m'].max():.0f}")
print("Checks passed.")
print(f"\nAnswer to the question of the session:")
print(f"  mean elevation of the ten points: {points['elev_m'].mean():.1f} m")
print(f"  minimum {points['elev_m'].min():.1f} m, maximum {points['elev_m'].max():.1f} m")

### Exercise 6. Zonal statistics *(if time allows)*

Sampling gives us the raster value at a specific point. Sometimes, however, we want to summarise a raster over an entire area.

A **zonal statistic** calculates a summary of raster values within a vector geometry. Examples include:

- mean elevation within a municipality
- accumulated rainfall within a catchment
- maximum slope within a project area

This operation connects the **field model** of the raster with the **object model** of the vector layer.

Use `rasterio.mask.mask` to calculate the mean elevation within each municipality and store the result in `mean_elev_m`.


In [ ]:
from rasterio.mask import mask

def mean_elev(geom, src):
    """Mean elevation inside a geometry, ignoring no-data cells."""
    # YOUR CODE HERE
    raise NotImplementedError

municipalities["mean_elev_m"] = [mean_elev(g, src) for g in municipalities.geometry]
municipalities[["nombre", "superficie", "mean_elev_m"]].round(1)

In [ ]:
assert municipalities["mean_elev_m"].notna().all()
assert municipalities["mean_elev_m"].between(400, 1200).all()
assert municipalities["mean_elev_m"].std() > 10, (
    "If all mean elevations are almost equal, the clipping is not working")
print("Checks passed.")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6.5))
municipalities.plot(column="mean_elev_m", cmap="terrain", legend=True, ax=ax,
                    edgecolor="white", linewidth=.8,
                    legend_kwds={"label": "mean elevation (m)", "shrink": .7})
points.plot(ax=ax, color="#E8590C", markersize=28, edgecolor="white", linewidth=.6)
ax.set_title("Mean elevation per municipality and traffic counters")
ax.set_axis_off(); plt.tight_layout()

### Extension A. One file, several vector layers

A GeoPackage (`.gpkg`) is a SQLite-based format that can store multiple vector layers in a single file.

Save:

- the municipalities as one layer
- the traffic measurement points as another layer

Then list the layers in the GeoPackage and compare its file size with the shapefile and GeoJSON versions.

The comparison is illustrative: the formats do not contain exactly the same information in this exercise, so file size alone should not be used to decide which format is preferable.


In [ ]:
# YOUR CODE HERE
raise NotImplementedError

print(gpd.list_layers(gpkg_path))
size_shp = sum(p.stat().st_size for p in shp_dir.iterdir()) / 1024
print(f"\nShapefile (5 files)   {size_shp:7.1f} kB")
print(f"GeoJSON               {(DATA/'municipios.geojson').stat().st_size/1024:7.1f} kB")
print(f"GeoPackage (2 layers) {gpkg_path.stat().st_size/1024:7.1f} kB")

### Extension B. From a raster surface to a table

A raster can also be converted into a tabular structure.

Select a small window of the terrain raster and create a table with **one row per raster cell**, including its elevation value.

Then plot the distribution of elevations.

After the conversion, consider:

> **What information is easier to analyse in the table, and what spatial structure has become less explicit?**

This exercise illustrates a broader idea discussed by Rey, Arribas-Bel and Wolf (2023): the same information can often be represented using different data structures, each of which makes some operations easier than others.


In [ ]:
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
src.close()
print("File closed.")

---

## Take-aways

By the end of this lab, you should be comfortable with six ideas:

- **Vector and raster data represent space differently.** Vector layers are well suited to discrete objects, while rasters represent values distributed across a surface.
- **A shapefile is a set of related files.** Losing one of its supporting files can remove information such as the CRS or text encoding.
- **Always check the CRS before combining spatial data.** Two valid layers can still be incompatible if their coordinates are expressed in different reference systems.
- **`set_crs()` and `to_crs()` are not interchangeable.** The first assigns CRS information; the second transforms coordinates.
- **Raster no-data values must be handled explicitly.** Like sentinel values in tables, they can distort calculations and visualisations if treated as real observations.
- **Sampling and zonal statistics connect vector and raster data.** Sampling extracts raster values at locations; zonal statistics summarise them within geographic objects.

The file format matters, but the underlying **data model, coordinate system and structure of the information** matter more.
